In [1]:
import logging
from utils.cross_validation_siamese_raw_multiple_datasets import CrossValidationSiameseMultipleDs
from utils.torch_siamese_raw import DATASET_PARAMETER_FILES, SKELETON_FORMATS


n_folds = 4
test_dataset_type = 'yolo_v26'
optimizer_type = "adamw"
batch_size = 64

max_epochs = 1000 

siamese_nn_types = ["conv1d", "transformer"]

learning_rates = [1e-3, 1e-4, 1e-5, 1e-6] 

diff_dataset_usages = ["MIX_ALL", "SKELETON_TYPE", "DATASET_TYPE"]


datasets = [
    ("MIX_ALL", ["yolo_v26"]),
    ("MIX_ALL", ["mocap"]),
    
    ("MIX_ALL", ["mocap", "yolo_v26"]),
    ("MIX_ALL", ["mocap", "yolo_v26", "openpose", "mediapipe"]),
    ("MIX_ALL", ["mocap", "yolo_v26", "vitpose", "openpose"]),
    ("MIX_ALL", ["mocap", "yolo_v26", "vitpose", "rtmpose", "hrnet" , "openpose", "mediapipe"]),
    
    ("SKELETON_TYPE", ["mocap", "yolo_v26", "vitpose", "openpose"]),
    ("SKELETON_TYPE", ["mocap", "yolo_v26", "vitpose", "rtmpose", "hrnet" , "openpose", "mediapipe"]),

    
    ("DATASET_TYPE", ["mocap", "yolo_v26"]),
    ("DATASET_TYPE", ["mocap", "yolo_v26", "openpose", "mediapipe"]),
    ("DATASET_TYPE", ["mocap", "yolo_v26", "vitpose", "openpose"]),
    ("DATASET_TYPE", ["mocap", "yolo_v26", "vitpose", "rtmpose", "hrnet" , "openpose", "mediapipe"]),
    
]

embedding_sizes = [16, 32, 64, 128]

use_butterworth_smoothed_bool = [False, True]

lr_codes = {1e-3:"1e-3", 1e-4:"1e-4", 1e-5:"1e-5", 1e-6:"1e-6"}

In [2]:
SKELETON_FORMATS

{'mocap': ['mocap'],
 'body_25': ['openpose'],
 'mediapipe': ['mediapipe'],
 'coco': ['hrnet',
  'rtmpose',
  'vitpose',
  'yolo_v11',
  'yolo_v26',
  'movenet_lightning',
  'movenet_thunder']}

In [10]:
import os

i = 0

errors = []
for siamese_nn_type in siamese_nn_types:
    for learning_rate in learning_rates:
        for diff_dataset_usage, selected_datasets in datasets:
            for embedding_size in embedding_sizes:
                for use_butterworth_smoothed in use_butterworth_smoothed_bool:
                    try:
                        use_butterworth_smoothed_flag = "__butterworth" if use_butterworth_smoothed else ""
                        selected_datasets_list = ""
                        for ds in selected_datasets:
                            selected_datasets_list += f"_{ds}"
                            
                        logger_name = f"{i}_{siamese_nn_type}_{lr_codes[learning_rate]}_{diff_dataset_usage}_embd{embedding_size}_{selected_datasets_list}{use_butterworth_smoothed_flag}"

                        if not os.path.exists(f"./json_reports/{logger_name}.json") and siamese_nn_type == "transformer" and learning_rate == 1e-3 and embedding_size == 32 and diff_dataset_usage == "SKELETON_TYPE":
                            print(logger_name)
                        
                            cv = CrossValidationSiameseMultipleDs(
                                log_level = logging.CRITICAL,
                                # log_level = logging.INFO,
                                logger_name = logger_name,
                            )
                            
                            cv.perform_training_with_rank_classification_and_multiple_datasets(
                                n_folds = n_folds,
                                test_dataset_type = test_dataset_type,
                                n_epochs = max_epochs,
                                batch_size = batch_size,
                                optimizer_type = optimizer_type,
                                show_plot = False,
                                
                                selected_datasets = selected_datasets,
                                diff_dataset_usage = diff_dataset_usage,
                                siamese_nn_type = siamese_nn_type,
                                learning_rate = learning_rate,
                                embedding_size = embedding_size,
                                use_butterworth_smoothed = use_butterworth_smoothed,
                            )
                    except Exception as e:
                        errors.append(logger_name)
                        print(f"!!!!! Error occurred: {e}")

                    i+=1

print(f"Errors: {errors}")

434_transformer_1e-3_SKELETON_TYPE_embd32__mocap_yolo_v26_vitpose_openpose
!!!!! Error occurred: Input contains NaN.
435_transformer_1e-3_SKELETON_TYPE_embd32__mocap_yolo_v26_vitpose_openpose__butterworth
442_transformer_1e-3_SKELETON_TYPE_embd32__mocap_yolo_v26_vitpose_rtmpose_hrnet_openpose_mediapipe


KeyboardInterrupt: 